# 12 · Final runs: corrected SUPPORT benchmark and numeric encoding

**Why**: the numeric parser had METABRIC's column layout hard-coded. On SUPPORT, five
physiological measurements entered the model unscaled and mean blood pressure was turned
into a constant — every earlier SUPPORT number on the numeric path (notebooks 03, 10) is
affected. METABRIC was parsed correctly.

* **Stage A** re-tunes the SUPPORT backbone on validation C_td (SurvTRACE's grid).
* **Stage B** re-runs the full SUPPORT benchmark: Cox, DeepHit, DSM, MENSA and our model
  with the loss/MMV variants, 5 splits.
* **Stage C** compares continuous (standardised), **quantile** (rank → normal) and
  discretised (10 bins) numeric input on METABRIC and SUPPORT, for our model and DeepHit.

Standalone: clones the code, needs Internet; GPU recommended.

## Options

In [ ]:
NAME = "12_final_support_and_encoding"

SEEDS = [0, 1, 2, 3, 4]          # seed s = init seed s AND split s (as in notebook 03)

# SUPPORT is re-run because the numeric parser had METABRIC's column layout hard-coded:
# five SUPPORT measurements were unscaled and mean blood pressure was lost. METABRIC was
# parsed correctly; its tuned backbone from notebook 03 is reused.
TUNE_SUPPORT = True
TUNE_GRID = {
    "transformer_num_hidden_layers": [2, 3, 4],
    "transformer_hidden_size": [8, 16],
    "transformer_intermediate_size": [32, 64],
    "learning_rate": [1e-3, 1e-4],
}
TUNED_METABRIC = ["transformer_num_hidden_layers=2", "transformer_hidden_size=16",
                  "transformer_intermediate_size=32", "learning_rate=0.001"]   # notebook 03

BASELINES = ["coxph", "deephit_paper", "dsm_paper", "mensa_paper"]
OUR_RECIPES = [("nllpch", None), ("nllpch_sample_ranking", None), ("nllpch_mmv", 0.01),
               ("nllpch_mmv_rank", 0.01), ("nllpch_mmv_rank", 0.1), ("nllpch_mmv_rank", 1.0)]

# Numeric encoding (paper §2.2): same model and splits, three encodings.
ENCODING_DATASETS = ["metabric", "support"]
ENCODING_MODELS = [("ours", "nllpch"), ("deephit_paper", None)]

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/version_1/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Datasets and encodings

In [ ]:
GROUP = {"metabric": "survtrace_metabric", "support": "survtrace_support"}
def enc_overrides(ds, enc):
    """(overrides, model-hub folder) for one dataset/encoding."""
    if enc == "continuous":
        return [f"dataset={ds}_numeric"], f"{ds}_numeric"
    if enc == "quantile":
        return [f"dataset={ds}_quantile", "data.parse.scale_method=quantile"], f"{ds}_quantile"
    return [f"data/load={ds}", f"data/parse={ds}", f"tokenizers={ds}", f"dataset={ds}"], ds   # discretised

if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    TUNE_GRID = {"transformer_num_hidden_layers": [2, 3]}
for ds in ENCODING_DATASETS:
    for enc in ["continuous", "quantile", "discretised"]:
        ov, _ = enc_overrides(ds, enc)
        r.prepare(f"{GROUP[ds]}/survival", ov, label=f"{ds}_{enc}")

## Stage A · tune the SUPPORT backbone (validation C_td only)

In [ ]:
import itertools
import pandas as pd
TUNED = {"metabric": TUNED_METABRIC, "support": []}
if TUNE_SUPPORT:
    keys = list(TUNE_GRID)
    grid = [dict(zip(keys, v)) for v in itertools.product(*TUNE_GRID.values())]
    ov, hub = enc_overrides("support", "continuous")
    runs = [Run(f"tune__support__{i:02d}", "finetune", "survtrace_support/survival",
                ov + [f"{k}={v}" for k, v in cfg.items()] + ["seed=0", "split_seed=0", "tasks/losses=nllpch"],
                hub, {"dataset": "support", "model": "tune", **cfg}) for i, cfg in enumerate(grid)]
    r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)
    val = r.collect(split="validation")
    val = val[(val["model"] == "tune") & (val["dataset"] == "support")].sort_values("ctd_weighted_avg", ascending=False)
    if len(val):
        best = val.iloc[0]
        TUNED["support"] = [f"{k}={best[k]:g}" for k in keys]
        display(val[keys + ["ctd_weighted_avg"]].head(5).rename(columns={"ctd_weighted_avg": "val C_td"}))
(r.results_dir / "tuned_backbone.json").write_text(json.dumps(TUNED, indent=2))
print("backbones:", TUNED)

## Stage B · SUPPORT benchmark (replaces notebook 03's SUPPORT half)

In [ ]:
runs = []
ov, hub = enc_overrides("support", "continuous")
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    base = {"dataset": "support", "seed": s, "stage": "benchmark"}
    for m in BASELINES:
        runs.append(Run(f"support__{m}__s{s}", "coxph" if m == "coxph" else "finetune",
                        f"survtrace_support/{m}", ov + so + ([] if m == "coxph" else TUNED["support"]),
                        hub, base | {"model": m}))
    for rec, lv in OUR_RECIPES:
        extra = [f"tasks/losses={rec}"] + ([f"mmv_variance_weight={lv}"] if lv is not None else [])
        tag = f"support__ours_{rec}" + (f"_lv{lv:g}" if lv is not None else "") + f"__s{s}"
        runs.append(Run(tag, "finetune", "survtrace_support/survival", ov + so + TUNED["support"] + extra,
                        hub, base | {"model": "ours", "recipe": rec, "lambda_v": lv}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Stage C · continuous vs quantile vs discretised input

In [ ]:
runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for ds in ENCODING_DATASETS:
        for enc in ["continuous", "quantile", "discretised"]:
            ov, hub = enc_overrides(ds, enc)
            base = {"dataset": f"{ds}_encoding", "seed": s, "repr": enc, "stage": "encoding"}
            for model, rec in ENCODING_MODELS:
                if model == "ours":
                    runs.append(Run(f"enc__{ds}_{enc}__ours_{rec}__s{s}", "finetune", f"{GROUP[ds]}/survival",
                                    ov + so + TUNED[ds] + [f"tasks/losses={rec}"], hub,
                                    base | {"model": "ours", "recipe": rec}))
                else:
                    runs.append(Run(f"enc__{ds}_{enc}__{model}__s{s}", "finetune", f"{GROUP[ds]}/{model}",
                                    ov + so + TUNED[ds], hub, base | {"model": model}))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
* `table_support`: the corrected SUPPORT benchmark (compare with SurvTRACE Table 2: Cox 0.566, DeepHit 0.609, DSM 0.615, PC-Hazard 0.626, SurvTRACE 0.640, averages over 25/50/75 %).
* `table_<ds>_encoding` + the paired encoding table below.

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

from scripts.runner import paired_vs
res = r.collect()
enc = res[res.get("stage") == "encoding"]
for metric in ["ctd_weighted_avg", "brier_survtrace_weighted_avg"]:
    t = paired_vs(enc, metric, by="repr", baseline="continuous", pair_on="seed", within=["dataset", "model"])
    t.to_csv(r.results_dir / "report" / f"encoding_paired_{metric}.csv", index=False)
    display(t)
hcols = ["ctd_0th_event_0.25", "ctd_0th_event_0.5", "ctd_0th_event_0.75"]
bench = res[res.get("stage") == "benchmark"]
if len(bench) and all(c in bench for c in hcols):
    from scripts.report import load
    lab = load([r.results_dir])
    lab = lab[lab.get("stage") == "benchmark"]
    g = lab.groupby("label")[hcols].agg(["mean", "std"]).round(3)
    g.to_csv(r.results_dir / "report" / "support_horizons.csv")
    display(g)

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)